# 02. Data Wrangling & Cleaning

**Objective:** Prepare the survey data for reliable analysis by checking duplicates, handling missing values and creating a normalised annual compensation measure.

## 1. Import libraries and load data

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DA0321EN-SkillsNetwork/LargeData/m1_survey_data.csv"
df = pd.read_csv(dataset_url)

## 2. Identify duplicate records

In [ ]:
duplicate_rows = df[df.duplicated()]
print(f"Duplicate rows: {len(duplicate_rows):,}")
duplicate_rows.head()

## 3. Remove duplicates

In [ ]:
df_clean = df.drop_duplicates().copy()
print(f"Original rows: {len(df):,}")
print(f"Rows after removing duplicates: {len(df_clean):,}")

## 4. Standardise missing values

The source data uses `?` in some fields. Convert these markers to proper missing values so Pandas can handle them consistently.

In [ ]:
df_clean.replace("?", np.nan, inplace=True)
missing_counts = df_clean.isna().sum().sort_values(ascending=False)
missing_counts.head(15)

## 5. Investigate missing values in WorkLoc

In [ ]:
df_clean["WorkLoc"].value_counts(dropna=False)

## 6. Impute missing WorkLoc values

Use the most frequent observed `WorkLoc` category as the replacement value.

In [ ]:
workloc_mode = df_clean["WorkLoc"].mode(dropna=True)[0]
df_clean["WorkLoc"] = df_clean["WorkLoc"].fillna(workloc_mode)
print(f"Imputation value: {workloc_mode}")
print(f"Remaining missing WorkLoc values: {df_clean["WorkLoc"].isna().sum():,}")

## 7. Normalise compensation

`CompFreq` indicates whether compensation is reported yearly, monthly or weekly. Convert each value to an annual equivalent so the figures can be compared on the same basis.

In [ ]:
df_clean["CompFreq"].value_counts(dropna=False)

In [ ]:
def normalize_compensation(row):
    if pd.isna(row["CompTotal"]) or pd.isna(row["CompFreq"]):
        return np.nan
    if row["CompFreq"] == "Weekly":
        return row["CompTotal"] * 52
    if row["CompFreq"] == "Monthly":
        return row["CompTotal"] * 12
    if row["CompFreq"] == "Yearly":
        return row["CompTotal"]
    return np.nan

df_clean["NormalizedAnnualCompensation"] = df_clean.apply(normalize_compensation, axis=1)
df_clean[["CompFreq", "CompTotal", "NormalizedAnnualCompensation"]].head()

In [ ]:
median_annual_compensation = df_clean["NormalizedAnnualCompensation"].median()
print(f"Median normalised annual compensation: {median_annual_compensation:,.2f}")

## Key takeaway

The cleaned dataset is now more suitable for downstream analysis because duplicate records have been removed, missing values have been treated explicitly, and compensation is expressed on a common annual basis.